This involves a take-home task on a toy dataset for you to demonstrate your capability to work on the research project. Please read the instructions below carefully. The task itself is straightforward and should not take more than 2 days to finish. That said, you will have 14 days to finish the task (until 7/16).
1. Your task is to predict the "stance" label of the 5751 tweets in the attached csv file. Specifically, your goal is to infer the stance of a tweet with respect to COVID-19 vaccination. The stance belongs to one of the three categories, "in-favor", "against", "neutral-or-unclear".
2. Data description
    - columns:
        -   "tweet": the text
        -   "label_true": the ground-truth label annotated by human raters.
        -   "label_pred": your predictions. Please fill your predictions into this column.
3. In order to generate the predictions, please use the following model.
    - https://huggingface.co/google/flan-t5-large
4. To feed the tweet into the model, you have to install the HuggingFace and PyTorch package in Python.
    - The instruction to feed the tweets to the model is described here. You should be able to run it on a CPU machine without a GPU.
        - https://huggingface.co/google/flan-t5-large#running-the-model-on-a-cpu
5. Critically, when you feed the tweet into the model, you should embed the tweet in a prompt such that the model can generate the prediction for the stance. One simple prompt you can use is as follows.
> What is the stance of the following tweet with respect to COVID-19 vaccine?  Here is the tweet. "{THE  }"  Please use exactly one word from the following 3 categories to label it: "in-favor", "against", "neutral-or-unclear".
6. Now, please predict the label "label_pred" in the csv file "Q2_20230202_majority.csv". Please keep in mind that the actual dataset consists of around 1 million of tweets, so you want to solve the program programitically.
7. Meanwhile, please push your code to your GitHub in a private repository (don't set it to public!).
8. Fine-tuning is highly encouraged. You may need GPU for fine-tuning. Consider using https://colab.research.google.com/ if you don't have other GPU compute. T4 GPU it provides for free should be enought to fine-tune a FLAN-T5-Large model.
9. After you finished the task, please (a) invite me as a collaborator to the repository. (b) share with me the link to your github repo and (c) send me the csv file with the columns "label_pred" filled with your predictions.
Your attempt will be evaluated based on the three criteria.
(1) The model F1 score on the held-out test set that you did not see.
(2) The readability of the codes, including comments, the use of OOP, etc. Please also include a README file at the repo's root directory.
(3) Whether you use Git control properly. This includes clear git commits etc.

In [1]:
!pip install -U transformers
import transformers
print(transformers.__version__)

4.53.2


In [ ]:
!pip install transformers datasets accelerate peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [16]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

df = pd.read_csv("Q2_20230202_majority 1.csv")
print("Data loaded into df!")

Data loaded into df!


In [ ]:
df.head()

In [5]:
from datasets import Dataset

dataset = Dataset.from_pandas(df[["tweet", "label_majority"]])
dataset = dataset.train_test_split(test_size=0.1)
print("Dataset loaded and split")

Dataset loaded and split


In [6]:
def preprocess(example):
    model_inputs = tokenizer(example["tweet"], truncation=True, padding="max_length", max_length=256)
    labels = tokenizer(example["label_majority"], truncation=True, padding="max_length", max_length=10)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs
print("Function laoded")

Function laoded


In [7]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")
tokenized_ds = dataset.map(preprocess, batched=True)

model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large")

# input_text = "translate English to German: How old are you?"
# input_ids = tokenizer(input_text, return_tensors="pt").input_ids

# outputs = model.generate(input_ids)
# print(tokenizer.decode(outputs[0]))

print("Tokenizer and model loaded!")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/576 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Tokenizer and model loaded!


In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# # During inference:
# inputs = tokenizer("Tweet: I'm happy I got the vaccine.", return_tensors="pt").to(device)
# outputs = model.generate(**inputs)
# print(tokenizer.decode(outputs[0], skip_special_tokens=True))


In [ ]:
print(device)

cuda


In [ ]:
from transformers import T5ForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./flan-t5-tweet-stance",
    eval_strategy="epoch",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,
    logging_dir='./logs',
    logging_steps=100
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer
)
print("Training...")
trainer.train()
print("model trained")

/tmp/ipython-input-10-1419398113.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Training...


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: contactmaanan (contactmaanan-university-of-wisconsin-madison) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,0.000000,nan
2,0.000000,nan
3,0.000000,nan


model trained


In [ ]:
trainer.save_model("./finetuned-flan-t5-large")
tokenizer.save_pretrained("./finetuned-flan-t5-large")

('./finetuned-flan-t5-large/tokenizer_config.json',
 './finetuned-flan-t5-large/special_tokens_map.json',
 './finetuned-flan-t5-large/spiece.model',
 './finetuned-flan-t5-large/added_tokens.json')

In [8]:
from textwrap import dedent

def build_prompt(tweet):
  # Tweet: "joe rogan: google is curating search engine results to hide info on vaccine-related injuries"
  #     Label: This tweet suggests that Google is actively suppressing information about vaccine-related injuries, which implies a distrust in vaccines or in public vaccine discourse. It frames the conversation negatively, aligning with an anti-vaccine stance.
  examples = dedent(
      """
      Tweet: "over 2.18 crore covid-19 vaccine doses still available with states, uts: govt | india news â€“ times ofÂ india"
      Label: 0
      Explanation: This tweet shares factual information about vaccine availability without expressing a clear opinion for or against the COVID-19 vaccine.

      Tweet: "1. all children matter &amp; we should be doing everything we can to protect them. .2. although most kids will be fine, previously healthy kids can and do get very sick with #covid19. ..#getvaccinated #vaccinateyourkids #wearamask #vaxu5"
      Label: 1
      Explanation: This tweet strongly supports COVID-19 vaccination, especially for children. It uses hashtags like #getvaccinated, #vaccinateyourkids, and #vaxu5, and emphasizes the importance of protecting children from COVID-19, showing clear pro-vaccine sentiment.


      """
      #USE ZERO SHOT, have it give rationale
  )

  prompt = dedent(
    f"""
    Given the following examples, determine the stance of tweets toward the COVID-19 vaccine:

    {examples}

    What is the stance of the following tweet?

    Tweet: {tweet}

    Label it as:
    1 for in-favor,
    -1 for against,
    0 neutral-or-unclear,

    """
  )


  prompt_zero_shot = dedent(
    f"""
    Given you're an expert medical journalist, please classify the stance of the following tweet toward the COVID-19 vaccine. [in-favor, against, neutral or unclear]
    What is the stance of the following tweet with respect to COVID-19 vaccination?

    Tweet: "{tweet}"

    Respond with a detailed rationale and label it as:
    1 for in-favor,
    -1 for against,
    0 neutral-or-unclear,

    Output format:
    Reasoning: [your reasoning]
    Label: [your label]
    """
  )
  return prompt_zero_shot

print("Function loaded")

Function loaded


In [9]:
def map_numeric_to_label(output):
    # output = output.strip().replace(" ", "")
    if output == "1":
        return "in-favor"
    elif output == "-1":
        return "against"
    else:
        return "neutral-or-unclear"

In [10]:
import re

def extract_label(output_text):
    """
    Extracts the label (-1, 0, 1) from the model's response text.
    Returns an int if found, else None.
    """
    match = re.search(r"Label\s*:\s*(-?1|0)", output_text)
    if match:
        return int(match.group(1))
    return None

In [11]:
def predict_stance(tweets, max_tokens=512):
    """
    Predicts the stance of a list of tweets using FLAN-T5.
    Returns both predicted labels and full model outputs.

    Args:
        tweets (list of str): List of tweet texts.
        max_tokens (int): Max tokens to generate.

    Returns:
        Tuple:
            - list of str: Predicted stance labels ("in-favor", "against", "neutral-or-unclear")
            - list of str: Full model output (reasoning + label)
    """
    prompts = [build_prompt(tweet) for tweet in tweets]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)

    outputs = model.generate(**inputs, max_new_tokens=max_tokens)
    decoded_outputs = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    predicted_labels = [
        map_numeric_to_label(extract_label(output)) for output in decoded_outputs
    ]

    return predicted_labels, decoded_outputs


In [ ]:
# def predict_stance(tweets, max_tokens):
#   prompts = [build_prompt(t) for t in tweets]
#   inputs  = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)
#   outputs = model.generate(**inputs, max_new_tokens=max_tokens)
#   labels  = tokenizer.batch_decode(outputs, skip_special_tokens=True)
#   return [map_numeric_to_label(label) for label in labels]

# print("Function loaded")

In [17]:
df = df.sample(50)

In [19]:
from tqdm import tqdm
from IPython.display import clear_output

BATCH = 8
MAX_TOKENS = 512
preds = []
print("Started...")


# for i in tqdm(range(0, len(df), BATCH)):

#   batch_tweets = df["tweet"].iloc[i : i + BATCH].tolist()
#   batch_preds = predict_stance(batch_tweets, MAX_TOKENS)
#   preds.extend(batch_preds)

#   clear_output(wait=True)
#   print(f"Processed {i + len(batch_preds)} tweets")
#   print(f"Latest batch output ({len(batch_preds)}): {batch_preds}")

# df["label_pred"] = preds



# Run predictions
preds, full_outputs = predict_stance(df["tweet"].tolist())

# Add both columns
df["label_pred"] = preds
df["model_output_raw"] = full_outputs


df.head(5)

Started...


,tweet_id,created_at,tweet,label_majority,month,label_pred,model_output_raw
4994,1.481799e+18,2022-01-14 01:24:14+00:00,"just a reminder that the supreme court made everyone involved in hearing the vaccine mandate case take a pcr test in advance, guaranteeing a level of protection that is wildly out of reach for nearly every worker affected by this decision",neutral-or-unclear,22-Jan,neutral-or-unclear,1 for in-favor
5684,1.534893e+18,2022-06-09 13:40:08+00:00,listening to lbc radio. people calling up saying they’re getting ‘stroke like symptoms’. ..all vaccinated...all blaming long covid...couldn’t make it up.,against,22-Jun,neutral-or-unclear,1 for in-favor
1516,1.455535e+18,2021-11-02 14:00:22+00:00,"for all those cops that take up a career in public safety, get your a** vaccinated..no one wants to hear your excuses..the vaccine have been out almost a year. that is plenty of time for you to of educated yourself. ..i’m with john oliver on this one:..#demvoice1 .#onev1",in-favor,21-Nov,neutral-or-unclear,1 for in-favor
4059,1.376375e+18,2021-03-29 03:24:31+00:00,"liberals create the s.....holes then leave ..to go do the same elsewhere..thats what we need passports for..not the vaccine,but liberals..",neutral-or-unclear,21-Mar,neutral-or-unclear,-1 for against
1045,1.442216e+18,2021-09-26 19:54:03+00:00,"what we don't know from the statistics is how effective was the vaccine at reducing symptoms for fully vaccinated who did get covid. the fact that many of these cases in the outbreak are at a personal care home, and yet there are only 2 people in hospital gives us a clue",in-favor,21-Sep,neutral-or-unclear,1 for in-favor


In [20]:
df["correct"] = df["label_majority"] == df["label_pred"]
accuracy = df["correct"].mean()
print(f"Raw accuracy: {accuracy:.2%}")

Raw accuracy: 16.00%


In [21]:
df.style.hide(axis="index")
pd.set_option('display.max_colwidth', None)

df[df["correct"] == False][["tweet", "label_majority", "label_pred"]]

,tweet,label_majority,label_pred
5684,listening to lbc radio. people calling up saying they’re getting ‘stroke like symptoms’. ..all vaccinated...all blaming long covid...couldn’t make it up.,against,neutral-or-unclear
1516,"for all those cops that take up a career in public safety, get your a** vaccinated..no one wants to hear your excuses..the vaccine have been out almost a year. that is plenty of time for you to of educated yourself. ..i’m with john oliver on this one:..#demvoice1 .#onev1",in-favor,neutral-or-unclear
1045,"what we don't know from the statistics is how effective was the vaccine at reducing symptoms for fully vaccinated who did get covid. the fact that many of these cases in the outbreak are at a personal care home, and yet there are only 2 people in hospital gives us a clue",in-favor,neutral-or-unclear
1435,"don't speak like idiot, at the peak time it's difficult to book slots because otp would never come fast. it's not modi money to say free vaccine",in-favor,neutral-or-unclear
1932,"however; they haven’t been affected by delta in the same way nb has from august on, because they didn’t drop all their restrictions and open their borders before target vax all at once, and they’ve reacted hard and fast to surges. they’ve fought outbreaks harder.",in-favor,neutral-or-unclear
4094,did someone say “vaccine queen”?! 💉💉💉,in-favor,neutral-or-unclear
4191,the u.s. covid-19 vaccine rollout is getting faster. but is it enough?,in-favor,neutral-or-unclear
3168,"hey #morningjoe. my immune compromised son caught covid outside. only time he took off his mask was to eat. he just accidentally stood too close to someone who was asymptomatic, and also maskless for eating. he’s ok, and we are all vaccinated now, but please keep distancing.",in-favor,neutral-or-unclear
2074,mandating shots for kids? i think you’re following the wrong ‘him’.,against,neutral-or-unclear
4445,we did so much clowning on nursing majors in college but look who's vaccinated now,in-favor,neutral-or-unclear


In [ ]:
df.sample(5)